### CSV And Excel files- Structured Data

In [3]:
import pandas as pd
import langchain
import os

In [4]:
os.makedirs("data/structured_files", exist_ok = True)

In [5]:
data = {
    "Product_ID": [101, 102, 103, 104, 105],
    "Product_Name": ["Dell Inspiron 15","Logitech Wireless Mouse","HP Mechanical Keyboard","Samsung 24-inch Monitor","Sony Bluetooth Headphones"],
    "Category": ["Laptop","Mouse","Keyboard","Monitor","Headphones"],"Brand": ["Dell","Logitech","HP","Samsung","Sony"],
    "Price": [55000,799,2499,12999,3499],
    "Stock": [15,100,50,25,40],
    "Rating": [4.5,4.3,4.4,4.6,4.7],
    "Description": [
        "15.6-inch laptop with Intel Core i5 processor, 8GB RAM, and 512GB SSD suitable for work and study.",
        "Ergonomic wireless mouse with adjustable DPI and long battery life.",
        "Mechanical keyboard with RGB lighting and durable switches for gaming and typing.",
        "24-inch Full HD monitor with IPS panel and slim bezel design.",
        "Wireless Bluetooth headphones with noise cancellation and up to 30 hours battery life."
    ],
    "Supplier": [
        "Tech Distributors Ltd",
        "Accessory World",
        "HP Partners",
        "Display Solutions",
        "Audio Hub"
    ]
}
df = pd.DataFrame(data)
df.to_csv("data/structured_files/products.csv", index=False)

In [6]:
with pd.ExcelWriter("data/structured_files/inventory.xlsx") as writer:
    df.to_excel(writer, sheet_name="Products", index=False)
    summary_data= {
        'Category': ['Electronics', 'Accessories'],
        'Total_Items': [3,2],
        'Total_Value': [1378.9, 1082.98]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

### CSV Processing

In [7]:
from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader

/tmp/ipykernel_63297/444435838.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader, UnstructuredCSVLoader
/home/jyoti-singh/Downloads/course-target/ragudemy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Method 1: CSVLOader - Row-based Documents
csv_loader = CSVLoader(
    file_path= "data/structured_files/products.csv",
    encoding="utf-8",
    csv_args={
        "delimiter": ",",
        "quotechar": '"',
    }
)
csv_docs = csv_loader.load()
print(f"Loaded {len(csv_docs)} documents (one per row)")
print("\n First Document:")
print(f"\n Content: {csv_docs[0].page_content}")
print(f"\n metadata: {csv_docs[0].metadata}")

Loaded 5 documents (one per row)

 First Document:

 Content: Product_ID: 101
Product_Name: Dell Inspiron 15
Category: Laptop
Brand: Dell
Price: 55000
Stock: 15
Rating: 4.5
Description: 15.6-inch laptop with Intel Core i5 processor, 8GB RAM, and 512GB SSD suitable for work and study.
Supplier: Tech Distributors Ltd

 metadata: {'source': 'data/structured_files/products.csv', 'row': 0}


In [9]:
# Method 2: Custom CSV processing for better control
from typing import List
from langchain_core.documents import Document
print("\n Custom CSV Processing")
def processing_csv_intelligently(filepath:str) -> List[Document]:
    """Processing CSV with intelligent document creation"""
    df = pd.read_csv(filepath)
    documents = []
    for idx, row in df.iterrows():
        content = f"""Product Information:
        Name: {row['Product_Name']}
        Category: {row['Category']}
        Price: ${row['Price']}
        Stock: {row['Stock']} units
        Description: {row['Description']}"""
        
    doc = Document(
        page_content = content,
        metadata={
            'source': filepath,
            'row_index': idx,
            'product_name': row['Product_Name'],
            'category': row['Category'],
            'price': row['Price'],
            'data_type': 'product_info'
        }
    )
    documents.append(doc)
    return documents


        


 Custom CSV Processing


In [10]:
processing_csv_intelligently('data/structured_files/products.csv')

[Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 4, 'product_name': 'Sony Bluetooth Headphones', 'category': 'Headphones', 'price': 3499, 'data_type': 'product_info'}, page_content='Product Information:\n        Name: Sony Bluetooth Headphones\n        Category: Headphones\n        Price: $3499\n        Stock: 40 units\n        Description: Wireless Bluetooth headphones with noise cancellation and up to 30 hours battery life.')]

In [19]:
# method1: Using pandas for full control
from typing import List
print("Pandas-based Excel Processing")
def process_excel_with_pandas(filepath:str) -> List[Document]:
    """Process Excel with Sheet awareness"""
    documents = []
    sheet_content = ""
    # read all sheets
    excel_file = pd.ExcelFile(filepath)
    for sheet_name in excel_file.sheet_names:
        df = pd.read_excel(filepath, sheet_name = sheet_name)
        sheet_content += f"Sheet: {sheet_name}\n"
        sheet_content += f"Coloumns: {", ".join(df.columns)}\n"
        sheet_content += f"Rows: {len(df)}\nn"
        sheet_content += df.to_string(index=False)
        doc = Document(
            page_content = sheet_content,
            metadata={
                'source': filepath,
                'sheet_name': sheet_name,
                'num_rows': len(df),
                'num_columns': len(df.columns),
                'data_type': 'excel_sheet'
            }
        )
        documents.append(doc)
    return documents
    

Pandas-based Excel Processing


In [20]:
excel_docs = process_excel_with_pandas("data/structured_files/inventory.xlsx")
print(f"Processed {len(excel_docs)} sheets")

Processed 2 sheets


In [21]:
excel_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Products', 'num_rows': 5, 'num_columns': 9, 'data_type': 'excel_sheet'}, page_content='Sheet: Products\nColoumns: Product_ID, Product_Name, Category, Brand, Price, Stock, Rating, Description, Supplier\nRows: 5\nn Product_ID              Product_Name   Category    Brand  Price  Stock  Rating                                                                                        Description              Supplier\n        101          Dell Inspiron 15     Laptop     Dell  55000     15     4.5 15.6-inch laptop with Intel Core i5 processor, 8GB RAM, and 512GB SSD suitable for work and study. Tech Distributors Ltd\n        102   Logitech Wireless Mouse      Mouse Logitech    799    100     4.3                                Ergonomic wireless mouse with adjustable DPI and long battery life.       Accessory World\n        103    HP Mechanical Keyboard   Keyboard       HP   2499     50     4.4                  

In [29]:
from langchain_community.document_loaders import UnstructuredExcelLoader

# Method 2 UnstructuredExcelLoader
print(f"\n UnstructuredExcelLoader")
try:
    
    excel_loader = UnstructuredExcelLoader(
        "data/structured_files/inventory.xlsx",
        mode="elements"
    )
    unstructured_docs = excel_loader.load()
    print(" Handles complex excel features")
    print(f"{unstructured_docs}")
    print("Preserves formatting info")
    print("Requires unstructured library")
except Exception as e:
    print("requires unstructured library with excel support")
    



 UnstructuredExcelLoader
 Handles complex excel features
[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-06-17T17:39:50', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product_ID</td><td>Product_Name</td><td>Category</td><td>Brand</td><td>Price</td><td>Stock</td><td>Rating</td><td>Description</td><td>Supplier</td></tr><tr><td>101</td><td>Dell Inspiron 15</td><td>Laptop</td><td>Dell</td><td>55000</td><td>15</td><td>4.5</td><td>15.6-inch laptop with Intel Core i5 processor, 8GB RAM, and 512GB SSD suitable for work and study.</td><td>Tech Distributors Ltd</td></tr><tr><td>102</td><td>Logitech Wireless Mouse</td><td>Mouse</td><td>Logitech</td><td>799</td><td>100</td><td>4.3</td><td>Ergonomic wireless mouse with adjustable DPI and long battery life.</td><td>Accessory World</td></tr><tr><td>103</td><td>HP Mechanical Keyboard</td><td>Keyboard</td>

In [30]:
unstructured_docs

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-06-17T17:39:50', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product_ID</td><td>Product_Name</td><td>Category</td><td>Brand</td><td>Price</td><td>Stock</td><td>Rating</td><td>Description</td><td>Supplier</td></tr><tr><td>101</td><td>Dell Inspiron 15</td><td>Laptop</td><td>Dell</td><td>55000</td><td>15</td><td>4.5</td><td>15.6-inch laptop with Intel Core i5 processor, 8GB RAM, and 512GB SSD suitable for work and study.</td><td>Tech Distributors Ltd</td></tr><tr><td>102</td><td>Logitech Wireless Mouse</td><td>Mouse</td><td>Logitech</td><td>799</td><td>100</td><td>4.3</td><td>Ergonomic wireless mouse with adjustable DPI and long battery life.</td><td>Accessory World</td></tr><tr><td>103</td><td>HP Mechanical Keyboard</td><td>Keyboard</td><td>HP</td><td>2499</td><td>50</td><td>4.4</td><td>Mechani